## 0. Kaggle bootstrap

Run this cell **first** on Kaggle. It clones the fork (for `src/`), installs
`pytorch-msssim` / `torchinfo`, links your uploaded `dirty.npy` / `clean.npy`
Dataset as `./data`, and `chdir`s into `notebooks/` so the rest of the
notebook's `../data` and `../results` paths resolve. On a local machine it is a no-op.

**Kaggle setup:** Settings → Accelerator = *GPU*, Internet = *On*; then
*Add Input* → your uploaded data Dataset.


In [ ]:
# >>> KAGGLE BOOTSTRAP >>>  (no-op when not running on Kaggle)
import os, sys, subprocess, glob

if os.path.exists('/kaggle'):
    REPO_URL = 'https://github.com/KrishanYadav333/EXXA.git'
    BRANCH   = 'week-4'
    REPO     = '/kaggle/working/EXXA'
    PKG      = os.path.join(REPO, 'DENOISING_DIFFUSION')  # contains src/, notebooks/

    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO], check=True)

    # deps not guaranteed on the Kaggle image
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim', 'torchinfo'], check=True)

    # expose the uploaded Kaggle Dataset as <repo>/data so '../data/*.npy' resolves
    os.makedirs(os.path.join(PKG, 'data'), exist_ok=True)
    hits = glob.glob('/kaggle/input/**/dirty.npy', recursive=True)
    if hits:
        src_dir = os.path.dirname(hits[0])
        for fn in ('dirty.npy', 'clean.npy'):
            dst = os.path.join(PKG, 'data', fn)
            if not os.path.exists(dst):
                try:
                    os.symlink(os.path.join(src_dir, fn), dst)
                except OSError:
                    import shutil; shutil.copy(os.path.join(src_dir, fn), dst)
    else:
        print('WARNING: dirty.npy not found under /kaggle/input — use *Add Input* to attach your data Dataset.')

    # run from notebooks/ so the existing '..'-relative paths work
    os.chdir(os.path.join(PKG, 'notebooks'))
    if PKG not in sys.path:
        sys.path.insert(0, PKG)

print('cwd :', os.getcwd())
# <<< KAGGLE BOOTSTRAP <<<

# 04 U-Net Model — Skip-Connection Denoiser for Protoplanetary Disks

Continues from `03_vae_model.ipynb`.  
We train the **DenoisingUNet** (3.4M params) — a lightweight U-Net with residual blocks,
GroupNorm, sinusoidal timestep conditioning, and **skip connections**.

Key design: for supervised denoising we set `t = 0` and use the same HybridLoss.


## 1. Setup and Imports

In [1]:
import sys, os, time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

from src.models.unet import DenoisingUNet
from src.utils.losses import HybridLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Device : cuda
GPU    : NVIDIA GeForce RTX 2050
VRAM   : 4.3 GB


In [2]:
# Load data — cast dirty to float32
dirty_all = np.load('../data/dirty.npy').astype(np.float32)
clean_all = np.load('../data/clean.npy').astype(np.float32)
print(f'dirty : {dirty_all.shape}  dtype={dirty_all.dtype}')
print(f'clean : {clean_all.shape}  dtype={clean_all.dtype}')

# 80/20 split
SEED = 42
indices = np.arange(len(dirty_all))
train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=SEED)
print(f'Train : {len(train_idx)}  Val : {len(val_idx)}')


dirty : (975, 600, 600)  dtype=float32
clean : (975, 600, 600)  dtype=float32
Train : 780  Val : 195


## 2. Model Summary

In [3]:
model_unet = DenoisingUNet(str(device))
total_params = sum(p.numel() for p in model_unet.parameters())
print(f'DenoisingUNet parameters : {total_params:,}')

# Forward pass check
x_test = torch.randn(2, 1, 64, 64).to(device)
t_test = torch.zeros(2, dtype=torch.long).to(device)
with torch.no_grad():
    out_test = model_unet(x_test, t_test)

print(f'Input  : {tuple(x_test.shape)}')
print(f'Output : {tuple(out_test.shape)}')
assert out_test.shape == (2, 1, 64, 64), f'Shape mismatch: {out_test.shape}'
print('Forward pass: OK')


DenoisingUNet parameters : 3,424,065


Input  : (2, 1, 64, 64)
Output : (2, 1, 64, 64)
Forward pass: OK


## 3. Training Loop ? 30 Epochs with Gradient Accumulation

| Setting | Value |
|---------|-------|
| Loss | `HybridLoss(alpha=0.8, beta=0.2)` |
| Optimizer | Adam, lr=1e-3 |
| Scheduler | ReduceLROnPlateau (factor 0.5, patience 5) |
| Batch | 16, grad accum ?4 ? effective 64 |
| Timestep | `t = 0` (supervised denoising, no diffusion noise) |

> The U-Net output is **unbounded** (no built-in sigmoid).  
> We apply `torch.sigmoid` before passing predictions into `HybridLoss`.


In [4]:
PATCH_SIZE = 64
BATCH_SIZE = 16
GRAD_ACCUM = 4
LR         = 1e-3
EPOCHS     = 30

np.random.seed(SEED); torch.manual_seed(SEED)

class PatchDataset(Dataset):
    def __init__(self, dirty, clean, idx, ps=64):
        self.dirty, self.clean, self.idx, self.ps = dirty, clean, idx, ps
        self._h, self._w = dirty.shape[1], dirty.shape[2]
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        idx = self.idx[i]
        r = np.random.randint(0, self._h - self.ps + 1)
        c = np.random.randint(0, self._w - self.ps + 1)
        dp = self.dirty[idx, r:r+self.ps, c:c+self.ps]
        cp = self.clean[idx, r:r+self.ps, c:c+self.ps]
        lo, hi = dp.min(), dp.max()
        if hi > lo:
            dp = (dp - lo) / (hi - lo)
            cp = np.clip((cp - lo) / (hi - lo), 0.0, 1.0)
        return torch.from_numpy(dp[np.newaxis]), torch.from_numpy(cp[np.newaxis])

train_loader = DataLoader(PatchDataset(dirty_all, clean_all, train_idx),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(PatchDataset(dirty_all, clean_all, val_idx),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')
print(f'Effective batch: {BATCH_SIZE * GRAD_ACCUM}')


Train batches: 49  Val batches: 13
Effective batch: 64


In [5]:
criterion = HybridLoss(alpha=0.8, beta=0.2)

# Fresh model + optimizer
model_unet = DenoisingUNet(str(device))
optimizer  = torch.optim.Adam(model_unet.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

best_val   = float('inf')
best_epoch = -1
train_hist = []
val_hist   = []

hdr = f"{'ep':>3}  {'tr_total':>10}  {'tr_mse':>10}  {'tr_ssim':>10}  {'val_total':>10}  {'lr':>8}"
print(hdr); print('-' * len(hdr))

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model_unet.train()
    tr_tot = tr_mse = tr_ssim = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (dirty, clean) in enumerate(train_loader, 1):
        dirty = dirty.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)
        t = torch.zeros(dirty.size(0), dtype=torch.long, device=device)

        pred = torch.sigmoid(model_unet(dirty, t))
        total, mse_l, ssim_l = criterion(pred, clean)
        (total / GRAD_ACCUM).backward()

        n = dirty.size(0)
        tr_tot  += total.item() * n
        tr_mse  += mse_l.item() * n
        tr_ssim += ssim_l.item()* n

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    N = len(train_loader.dataset)
    tr_tot /= N; tr_mse /= N; tr_ssim /= N

    # Validation
    model_unet.eval()
    vl = 0.0
    with torch.no_grad():
        for dirty, clean in val_loader:
            dirty = dirty.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)
            t = torch.zeros(dirty.size(0), dtype=torch.long, device=device)
            pred = torch.sigmoid(model_unet(dirty, t))
            total, _, _ = criterion(pred, clean)
            vl += total.item() * dirty.size(0)
    vl /= len(val_loader.dataset)

    scheduler.step(vl)
    lr_now = optimizer.param_groups[0]['lr']
    train_hist.append(tr_tot); val_hist.append(vl)

    mark = ' <<' if vl < best_val else ''
    if vl < best_val:
        best_val = vl; best_epoch = epoch
        best_state = {k: v.clone() for k, v in model_unet.state_dict().items()}
        os.makedirs('../results/checkpoints', exist_ok=True)
        torch.save({
            'epoch': best_epoch,
            'model_state_dict': model_unet.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val,
            'alpha': 0.8, 'beta': 0.2,
            'arch': 'DenoisingUNet',
            'base_channels': 32, 'channel_multipliers': [1, 2, 4],
        }, '../results/checkpoints/unet_best.pth')

    print(f'{epoch:>3}  {tr_tot:>10.6f}  {tr_mse:>10.6f}  {tr_ssim:>10.6f}  '
          f'{vl:>10.6f}  {lr_now:>8.2e}  {time.time()-t0:.1f}s{mark}')

print(f'
Best val loss : {best_val:.6f} at epoch {best_epoch}')

# Reload best weights
model_unet.load_state_dict(best_state)
model_unet.eval()
print('Checkpoint saved -> ../results/checkpoints/unet_best.pth')


 ep    tr_total      tr_mse     tr_ssim   val_total        lr
-------------------------------------------------------------


  1    0.172614    0.051706    0.656246    0.143486  1.00e-03  6.5s <<


  2    0.115539    0.022074    0.489400    0.097506  1.00e-03  5.7s <<


  3    0.076775    0.014302    0.326667    0.058240  1.00e-03  5.6s <<


  4    0.069121    0.014591    0.287242    0.067390  1.00e-03  5.5s


...

 26    0.042183    0.009061    0.174675    0.042915  1.00e-03  5.8s


 27    0.042300    0.009277    0.174391    0.037390  1.00e-03  5.7s


 28    0.044908    0.009274    0.187442    0.046661  1.00e-03  5.6s


 29    0.042923    0.008632    0.180085    0.042984  5.00e-04  5.8s


 30    0.040719    0.009267    0.166524    0.036962  5.00e-04  6.0s

Best val loss : 0.036899 at epoch 23
Checkpoint saved -> ../results/checkpoints/unet_best.pth


## 4. Loss Curves

In [6]:
fig, ax = plt.subplots(figsize=(9, 5))
eps = range(1, EPOCHS + 1)
ax.plot(eps, train_hist, label='Train Hybrid Loss', linewidth=2, color='#4C9EEB')
ax.plot(eps, val_hist,   label='Val Hybrid Loss',   linewidth=2, color='#E8715A', linestyle='--')
ax.axvline(best_epoch, color='gray', linestyle=':', linewidth=1.2,
           label=f'Best epoch ({best_epoch})')
ax.scatter([best_epoch], [best_val], color='#E8715A', zorder=5, s=80)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Hybrid Loss (0.8×MSE + 0.2×SSIM)', fontsize=10)
ax.set_title('DenoisingUNet — Training Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/unet_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> ../results/unet_loss.png')


Saved -> ../results/unet_loss.png


C:\Users\kk456\AppData\Local\Temp\ipykernel_15512\2853304011.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Visual Comparison ? with per-patch SSIM

3 random validation samples with 5 columns: **Clean Ground Truth | Noisy Input | Autoencoder Hybrid output | VAE output | U-Net output**.  
SSIM score is printed below each model output column.


In [7]:
from src.models.autoencoder import DenoisingAutoencoder
from src.models.vae import DenoisingVAE
from skimage.metrics import structural_similarity as ssim_fn

# Load AE hybrid checkpoint
model_ae = DenoisingAutoencoder().to(device)
ck_ae    = torch.load('../results/checkpoints/autoencoder_hybrid_best.pth', map_location=device)
model_ae.load_state_dict(ck_ae['model_state_dict']); model_ae.eval()

# Load VAE checkpoint
model_vae = DenoisingVAE(latent_dim=128).to(device)
ck_vae    = torch.load('../results/checkpoints/vae_best.pth', map_location=device)
model_vae.load_state_dict(ck_vae['model_state_dict']); model_vae.eval()

# U-Net already loaded with best_state above
print(f'AE Hybrid  : epoch {ck_ae.get("epoch","?")}')
print(f'VAE        : epoch {ck_vae["epoch"]}')
print(f'U-Net      : epoch {best_epoch}')

rng = np.random.default_rng(99)
ids = rng.choice(val_idx, size=3, replace=False)
r, c = 268, 268

cols = ['Clean Ground Truth', 'Noisy Input', 'Autoencoder Hybrid', 'VAE', 'U-Net']
fig, axes = plt.subplots(3, 5, figsize=(18, 11.8))
for col_idx, title in enumerate(cols):
    axes[0, col_idx].set_title(title, fontsize=12, fontweight='bold', pad=10)

for row, idx in enumerate(ids):
    dp = dirty_all[idx, r:r+64, c:c+64]
    cp = clean_all[idx, r:r+64, c:c+64]
    lo, hi = dp.min(), dp.max()
    dp_n = (dp - lo) / (hi - lo) if hi > lo else dp
    cp_n = np.clip((cp - lo) / (hi - lo), 0, 1) if hi > lo else cp

    inp = torch.from_numpy(dp_n[np.newaxis, np.newaxis]).to(device)
    t0_ = torch.zeros(1, dtype=torch.long, device=device)

    with torch.no_grad():
        pred_ae  = model_ae(inp).squeeze().cpu().numpy()
        pred_vae = model_vae(inp)[0].squeeze().cpu().numpy()
        pred_un  = torch.sigmoid(model_unet(inp, t0_)).squeeze().cpu().numpy()

    images = [cp_n, dp_n, pred_ae, pred_vae, pred_un]
    for col_idx, img in enumerate(images):
        ax = axes[row, col_idx]
        ax.imshow(np.clip(img, 0, 1), cmap='inferno', vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if col_idx >= 2:
            s = ssim_fn(cp_n, np.clip(img, 0, 1), data_range=1.0)
            ax.set_xlabel(f'SSIM={s:.4f}', fontsize=10, labelpad=7, color='black')
        else:
            ax.set_xlabel('')

fig.suptitle('Clean Ground Truth | Noisy Input | Autoencoder Hybrid | VAE | U-Net - 64x64 centre crop',
             fontweight='bold', fontsize=14, y=0.985)
fig.subplots_adjust(left=0.02, right=0.995, top=0.94, bottom=0.055, wspace=0.06, hspace=0.20)
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/unet_vs_all_comparison.png', dpi=150)
plt.show()
print('Saved -> ../experiments/unet_vs_all_comparison.png')


AE Hybrid  : epoch 27
VAE        : epoch 27
U-Net      : epoch 23


Saved -> ../experiments/unet_vs_all_comparison.png


C:\Users\kk456\AppData\Local\Temp\ipykernel_15512\4093695662.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Unified Metrics Table — All 8 Methods on 100 Val Samples

Evaluates all methods on the same 100 validation images.  
Results saved to `results/metrics_week3_final.csv`.


In [8]:
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import wiener as wiener_filter
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

PATCH_SZ = 64; STRIDE = 32; BATCH_INF = 32; N_EVAL = 100

_, val_idx_e = train_test_split(np.arange(len(dirty_all)), test_size=0.20, random_state=SEED)
rng_e   = np.random.default_rng(SEED)
chosen  = np.sort(rng_e.choice(val_idx_e, size=min(N_EVAL, len(val_idx_e)), replace=False))
print(f'Evaluating {len(chosen)} val samples  (idx {chosen[0]}..{chosen[-1]})')

# ── Sliding-window neural inference ───────────────────────────────────────────
def nn_denoise(mdl, img, use_sigmoid=False, is_vae=False):
    H, W = img.shape
    out_s = np.zeros((H,W), np.float64); out_c = np.zeros((H,W), np.float64)
    rows = list(range(0, H-PATCH_SZ+1, STRIDE)); cols_ = list(range(0, W-PATCH_SZ+1, STRIDE))
    if rows[-1]+PATCH_SZ < H: rows.append(H-PATCH_SZ)
    if cols_[-1]+PATCH_SZ < W: cols_.append(W-PATCH_SZ)
    patches, positions = [], []
    for rr in rows:
        for cc in cols_:
            p = img[rr:rr+PATCH_SZ, cc:cc+PATCH_SZ].copy()
            lo, hi = p.min(), p.max()
            p = (p-lo)/(hi-lo) if hi>lo else p
            patches.append(p); positions.append((rr,cc))
    mdl.eval(); preds = []
    with torch.no_grad():
        for s in range(0, len(patches), BATCH_INF):
            b = torch.from_numpy(np.stack(patches[s:s+BATCH_INF])[:,np.newaxis]).to(device)
            if is_vae:
                o = mdl(b)[0]
            elif use_sigmoid:
                t_b = torch.zeros(b.size(0), dtype=torch.long, device=device)
                o = torch.sigmoid(mdl(b, t_b))
            else:
                o = mdl(b)
            preds.extend(o.squeeze(1).cpu().numpy())
    for pred, (rr, cc) in zip(preds, positions):
        out_s[rr:rr+PATCH_SZ, cc:cc+PATCH_SZ] += pred
        out_c[rr:rr+PATCH_SZ, cc:cc+PATCH_SZ] += 1.0
    return np.clip(out_s / np.maximum(out_c, 1e-8), 0, 1).astype(np.float32)

# ── Also load MSE-only AE ────────────────────────────────────────────────────
model_ae_mse = DenoisingAutoencoder().to(device)
ck_ae_mse    = torch.load('../results/checkpoints/autoencoder_best.pth', map_location=device)
model_ae_mse.load_state_dict(ck_ae_mse['model_state_dict']); model_ae_mse.eval()

# ── Evaluate all 8 methods ───────────────────────────────────────────────────
methods = ['Noisy Input', 'Gaussian s=2', 'Median 3x3', 'Wiener',
           'AE MSE-only', 'AE HybridLoss', 'VAE (MSE+SSIM+KL)', 'U-Net HybridLoss']
acc = {m: {'PSNR': [], 'SSIM': [], 'MSE': []} for m in methods}

t_start = time.time()
for i, idx in enumerate(chosen):
    c_img = clean_all[idx]; d_img = dirty_all[idx]
    def push(name, arr):
        d = np.clip(arr.astype(np.float32), 0, 1)
        acc[name]['PSNR'].append(psnr_fn(c_img, d, data_range=1.0))
        acc[name]['SSIM'].append(ssim_fn(c_img, d, data_range=1.0))
        acc[name]['MSE'].append(float(np.mean((c_img - d)**2)))

    push('Noisy Input',        d_img)
    push('Gaussian s=2',       gaussian_filter(d_img, sigma=2.0))
    push('Median 3x3',         median_filter(d_img, size=3))
    push('Wiener',             wiener_filter(d_img).astype(np.float32))
    push('AE MSE-only',        nn_denoise(model_ae_mse, d_img))
    push('AE HybridLoss',      nn_denoise(model_ae,     d_img))
    push('VAE (MSE+SSIM+KL)',  nn_denoise(model_vae,    d_img, is_vae=True))
    push('U-Net HybridLoss',   nn_denoise(model_unet,   d_img, use_sigmoid=True))

    if (i+1) % 25 == 0:
        print(f'  [{i+1:>3}/100]  {time.time()-t_start:.0f}s')

print(f'Done in {time.time()-t_start:.1f}s')

# ── Build clean DataFrame ────────────────────────────────────────────────────
rows = []
for m in methods:
    rows.append({
        'Method': m,
        'PSNR': round(float(np.mean(acc[m]['PSNR'])), 4),
        'SSIM': round(float(np.mean(acc[m]['SSIM'])), 4),
        'MSE':  round(float(np.mean(acc[m]['MSE'])),  6),
    })
df = pd.DataFrame(rows).sort_values('SSIM', ascending=False).reset_index(drop=True)
df.index += 1
df.index.name = 'Rank'

csv_path = '../results/metrics_week3_final.csv'
df.to_csv(csv_path)
print(f'\nSaved -> {csv_path}')

# ── Print table ───────────────────────────────────────────────────────────────
best_p = df['PSNR'].max(); best_s = df['SSIM'].max(); best_m = df['MSE'].min()
print(f"\n{'='*64}")
print(f"  Week 3 Final Leaderboard — {len(chosen)} Val Samples  (ranked by SSIM)")
print(f"{'='*64}")
print(f"  {'Rank':<5} {'Method':<24} {'PSNR':>8}  {'SSIM':>6}  {'MSE':>10}")
print(f"  {'-'*60}")
for rank, row in df.iterrows():
    p_s = ' *' if row['PSNR']==best_p else '  '
    s_s = ' *' if row['SSIM']==best_s else '  '
    m_s = ' *' if row['MSE'] ==best_m else '  '
    print(f"  {rank:<5} {row['Method']:<24} {row['PSNR']:>8.4f}{p_s} {row['SSIM']:>6.4f}{s_s} {row['MSE']:>10.6f}{m_s}")
print(f"{'='*64}")
print(f"  * = best in column")


Evaluating 100 val samples  (idx 29..973)


  [ 25/100]  44s


  [ 50/100]  94s


  [ 75/100]  141s


  [100/100]  187s
Done in 186.7s

Saved -> ../results/metrics_week3_final.csv

  Week 3 Final Leaderboard — 100 Val Samples  (ranked by SSIM)
  Rank  Method                       PSNR    SSIM         MSE
  ------------------------------------------------------------
  1     AE HybridLoss             19.9152   0.7609 *   0.013920  
  2     VAE (MSE+SSIM+KL)         19.9951   0.7059     0.013336  
  3     U-Net HybridLoss          20.6345   0.7044     0.011653  
  4     AE MSE-only               20.2513   0.6158     0.012804  
  5     Gaussian s=2              22.7803   0.4230     0.006380  
  6     Median 3x3                22.8835 * 0.3591     0.006317 *
  7     Wiener                    22.5687   0.3398     0.006702  
  8     Noisy Input               21.5703   0.1924     0.008391  
  * = best in column


## Summary

### Architecture comparison

| Model | Params | Skip Connections | Probabilistic | DDPM backbone |
|-------|--------|:---:|:---:|:---:|
| Autoencoder | 1.73M | ❌ | ❌ | ❌ |
| VAE | ~3.2M | ❌ | ✅ | ❌ |
| **U-Net** | **3.4M** | **✅** | ❌ | **✅** |

### What skip connections add
- Encoder features concatenated into decoder → preserves fine edges
- Residual blocks + GroupNorm → faster convergence than plain ConvBlocks
- Timestep conditioning → ready for DDPM noise prediction

### Next steps
- Wire U-Net as ε_θ in DDPM diffusion loop
- Integrate NoiseScheduler for multi-step denoising


## Project Progress Summary

**Week 2 completed:** dataset exploration, classical baselines, Autoencoder MSE, Autoencoder Hybrid, and VAE.

**Week 3 completed:** U-Net with hybrid loss.

**Best model so far:** Autoencoder Hybrid has the highest SSIM, with **SSIM = 0.7609** on the 100-sample validation leaderboard.

**Next step:** diffusion model in Week 4.
